In [1]:
import ibis
import duckdb as db
import pandas as pd
from dataclasses import dataclass
import sys
sys.path.append("../")
from sql_ai_agent.parse_query import is_markdown_code_chunk, extract_code_from_markdown
from sql_ai_agent2.db_handler import get_tbl_attr

In [2]:
ibis.duckdb.connect()

TODO:
- Modify the prompt templates to LangChain

In [3]:
tbl_name = "air_traffic"

In [4]:
import ibis
print(ibis.__version__)
print(ibis.util.backend_entry_points())

11.0.0
[EntryPoint(name='athena', value='ibis.backends.athena', group='ibis.backends'), EntryPoint(name='bigquery', value='ibis.backends.bigquery', group='ibis.backends'), EntryPoint(name='clickhouse', value='ibis.backends.clickhouse', group='ibis.backends'), EntryPoint(name='databricks', value='ibis.backends.databricks', group='ibis.backends'), EntryPoint(name='datafusion', value='ibis.backends.datafusion', group='ibis.backends'), EntryPoint(name='druid', value='ibis.backends.druid', group='ibis.backends'), EntryPoint(name='duckdb', value='ibis.backends.duckdb', group='ibis.backends'), EntryPoint(name='exasol', value='ibis.backends.exasol', group='ibis.backends'), EntryPoint(name='flink', value='ibis.backends.flink', group='ibis.backends'), EntryPoint(name='impala', value='ibis.backends.impala', group='ibis.backends'), EntryPoint(name='mssql', value='ibis.backends.mssql', group='ibis.backends'), EntryPoint(name='mysql', value='ibis.backends.mysql', group='ibis.backends'), EntryPoint(n

In [5]:
# Check if the postgres backend module exists
try:
    import ibis.backends.postgres
    print("postgres backend module found")
except ImportError as e:
    print(f"postgres backend module missing: {e}")

# Check psycopg
try:
    import psycopg2
    print(f"psycopg2 version: {psycopg2.__version__}")
except ImportError:
    print("psycopg2 not found")
    
try:
    import psycopg
    print(f"psycopg3 version: {psycopg.__version__}")
except ImportError:
    print("psycopg3 not found")

postgres backend module found
psycopg2 not found
psycopg3 version: 3.2.12


In [5]:
con_ibis = ibis.postgres.connect(
    user="postgres",
    password="password",
    host="postgres",
    port=5432,
    database="my_db",
)

In [6]:
con_ibis.name

'postgres'

In [8]:
air_traffic = con_ibis.sql("SELECT * FROM air_traffic").execute()

air_traffic.head()

,Activity Period,Activity Period Start Date,Operating Airline,Operating Airline IATA Code,Published Airline,Published Airline IATA Code,GEO Summary,GEO Region,Activity Type Code,Price Category Code,Terminal,Boarding Area,Passenger Count,data_as_of,data_loaded_at
0,199907,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Deplaned,Low Fare,Terminal 1,B,31432,2025/09/20 01:01:11 PM,2025/09/22 03:10:03 PM
1,199907,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Enplaned,Low Fare,Terminal 1,B,31353,2025/09/20 01:01:11 PM,2025/09/22 03:10:03 PM
2,199907,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Thru / Transit,Low Fare,Terminal 1,B,2518,2025/09/20 01:01:11 PM,2025/09/22 03:10:03 PM
3,199907,1999-07-01,Aeroflot Russian International Airlines,None,Aeroflot Russian International Airlines,None,International,Europe,Deplaned,Other,Terminal 2,D,1324,2025/09/20 01:01:11 PM,2025/09/22 03:10:03 PM
4,199907,1999-07-01,Aeroflot Russian International Airlines,None,Aeroflot Russian International Airlines,None,International,Europe,Enplaned,Other,Terminal 2,D,1198,2025/09/20 01:01:11 PM,2025/09/22 03:10:03 PM


In [9]:
tbl_name = "air_traffic"
con_db = ibis.duckdb.connect()
print(con_db.name)
con_db.create_table(tbl_name, air_traffic)

duckdb


DatabaseTable: memory.main.air_traffic
  Activity Period             int64
  Activity Period Start Date  timestamp(6)
  Operating Airline           string
  Operating Airline IATA Code string
  Published Airline           string
  Published Airline IATA Code string
  GEO Summary                 string
  GEO Region                  string
  Activity Type Code          string
  Price Category Code         string
  Terminal                    string
  Boarding Area               string
  Passenger Count             int64
  data_as_of                  string
  data_loaded_at              string

In [10]:
# schema_ibis =  get_schema(con = con_ibis, tbl_name = tbl_name)
# print(schema_ibis)
schema_db = get_tbl_attr(con = con_db, tbl_name = tbl_name)
print(schema_db.schema)

print(schema_db.table)


Activity Period BIGINT, Activity Period Start Date TIMESTAMP, Operating Airline VARCHAR, Operating Airline IATA Code VARCHAR, Published Airline VARCHAR, Published Airline IATA Code VARCHAR, GEO Summary VARCHAR, GEO Region VARCHAR, Activity Type Code VARCHAR, Price Category Code VARCHAR, Terminal VARCHAR, Boarding Area VARCHAR, Passenger Count BIGINT, data_as_of VARCHAR, data_loaded_at VARCHAR
                    column_name column_type
0               Activity Period      BIGINT
1    Activity Period Start Date   TIMESTAMP
2             Operating Airline     VARCHAR
3   Operating Airline IATA Code     VARCHAR
4             Published Airline     VARCHAR
5   Published Airline IATA Code     VARCHAR
6                   GEO Summary     VARCHAR
7                    GEO Region     VARCHAR
8            Activity Type Code     VARCHAR
9           Price Category Code     VARCHAR
10                     Terminal     VARCHAR
11                Boarding Area     VARCHAR
12              Passenger Count 

In [11]:
schema_ibis =  get_tbl_attr(con = con_ibis, tbl_name = tbl_name)
print(schema_ibis.schema)
print(schema_ibis.table) 

Activity Period bigint, Activity Period Start Date timestamp without time zone, Operating Airline character varying, Operating Airline IATA Code character varying, Published Airline character varying, Published Airline IATA Code character varying, GEO Summary character varying, GEO Region character varying, Activity Type Code character varying, Price Category Code character varying, Terminal character varying, Boarding Area character varying, Passenger Count bigint, data_as_of character varying, data_loaded_at character varying
                    column_name                  column_type
0               Activity Period                       bigint
1    Activity Period Start Date  timestamp without time zone
2             Operating Airline            character varying
3   Operating Airline IATA Code            character varying
4             Published Airline            character varying
5   Published Airline IATA Code            character varying
6                   GEO Summary        

In [12]:
from langchain_core.prompts import (
  SystemMessagePromptTemplate,
  HumanMessagePromptTemplate,
  ChatPromptTemplate
)

In [13]:
schema_ibis.schema

'Activity Period bigint, Activity Period Start Date timestamp without time zone, Operating Airline character varying, Operating Airline IATA Code character varying, Published Airline character varying, Published Airline IATA Code character varying, GEO Summary character varying, GEO Region character varying, Activity Type Code character varying, Price Category Code character varying, Terminal character varying, Boarding Area character varying, Passenger Count bigint, data_as_of character varying, data_loaded_at character varying'

In [14]:
from sql_ai_agent2.prompt_handler import set_prompt

prompt = set_prompt(tbl_name = tbl_name, 
                    schema = schema_ibis.schema, 
                    additional_context = "", 
                    question =  "test" )



print(prompt)
print(prompt.messages[0].content)

messages=[SystemMessage(content="Given the following SQL table, your job is to write queries given a user’s request.\nReturn just the SQL query as plain text, without additional text, and don't use markdown format.\n\nCREATE TABLE air_traffic (Activity Period bigint, Activity Period Start Date timestamp without time zone, Operating Airline character varying, Operating Airline IATA Code character varying, Published Airline character varying, Published Airline IATA Code character varying, GEO Summary character varying, GEO Region character varying, Activity Type Code character varying, Price Category Code character varying, Terminal character varying, Boarding Area character varying, Passenger Count bigint, data_as_of character varying, data_loaded_at character varying)", additional_kwargs={}, response_metadata={}), HumanMessage(content='Write a SQL query that returns: test', additional_kwargs={}, response_metadata={})]
Given the following SQL table, your job is to write queries given a 

In [ ]:
question = "test"
from langchain_core.prompts import ChatPromptTemplate

system_template = """"
        Given the following SQL table, your job is to write queries given a user’s request. 
        Return just the SQL query as plain text, without additional text, and don't use markdown format. {additional_context}
        CREATE TABLE {tbl_name} ({schema})
        """

# system_template.format(tbl_name = tbl_name, schema = schema_db.schema)

user_template = "Write a SQL query that returns: {question}"

messages = [
    ("system", system_template),
    ("user", user_template)
]


prompt_template = ChatPromptTemplate.from_messages(messages)

prompt = prompt_template.invoke({"tbl_name": tbl_name, "schema": schema_ibis.schema, "additional_context": "", "question": question })

print(prompt)
print(prompt.messages[0].content)

messages=[SystemMessage(content='"\n        Given the following SQL table, your job is to write queries given a user’s request. \n        Return just the SQL query as plain text, without additional text, and don\'t use markdown format. \n        CREATE TABLE air_traffic (Activity Period bigint, Activity Period Start Date timestamp without time zone, Operating Airline character varying, Operating Airline IATA Code character varying, Published Airline character varying, Published Airline IATA Code character varying, GEO Summary character varying, GEO Region character varying, Activity Type Code character varying, Price Category Code character varying, Terminal character varying, Boarding Area character varying, Passenger Count bigint, data_as_of character varying, data_loaded_at character varying)\n        ', additional_kwargs={}, response_metadata={}), HumanMessage(content='Write a SQL query that returns: test', additional_kwargs={}, response_metadata={})]
"
        Given the following 

In [ ]:
from sql_ai_agent2.db_handler import get_tbl_attr




@dataclass
class SystemPrompt:
    system: str
    schema: str
    col_names: str
    col_types: str
    tbl_name: str



chat_template = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template(system),
    ]
)



def system_prompt(tbl_name, con):
    # Get table schema
    tbl_attr = get_tbl_attr(tbl_name=tbl_name, con = con)


    # Prompt templates
    system_template = (
        "Given the following SQL table, your job is to write queries given a user’s request. "
        "Return just the SQL query as plain text, without additional text, and don't use markdown format.\n\n"
        f"CREATE TABLE {tbl_name} ({tbl_attr.schema})\n"
    )
    return SystemPrompt(
        system=system_template,
        schema=tbl_attr.schema,
        col_names=tbl_attr.table["column_name"],
        col_types=tbl_attr.table["column_type"],
        tbl_name=tbl_name,
    )


def user_prompt(question):
    user_template = f"Write a SQL query that returns: {question}"
    return user_template


In [ ]:
sys_prompt = system_prompt(tbl_name = tbl_name, con = con_ibis)
sys_prompt.system

In [ ]:
from openai import OpenAI
class SqlAgent:
    def __init__(self, con, api_key, base_url, model, tbl_name, max_token=5000):
        self.api_key = api_key
        self.base_url = base_url
        self.model = model
        self.tbl_name = tbl_name
        self.max_token = max_token

        self.client = OpenAI(api_key=api_key, base_url=base_url)
        self.system = system_prompt(tbl_name=tbl_name)

    def send_prompt(self, question):
        self.user = user_prompt(question=question)
        self.response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": self.system.system},
                {"role": "user", "content": self.user},
            ],
            max_completion_tokens=self.max_token,
        )

        content = self.response.choices[0].message.content
        if is_markdown_code_chunk(text=content):
            query = extract_code_from_markdown(markdown_text=content)
        else:
            query = content

        self.query = query

    def ask_question(self, question, verbose=True):
        self.send_prompt(question=question)
        self.data = con.sql(self.query).execute()
        if verbose:
            print(self.query)
            print(self.data)